In [1]:
import numpy as np
import faiss
import sqlite3
import pandas as pd

In [2]:
# Connexion à la base de données SQLite
conn = sqlite3.connect('../data/Y-love.db')  # Se connecter à la base de données
cur = conn.cursor()  # Créer un curseur

In [3]:
def moyenne_ponderee_embeddings(vecs, poids=None):
    if poids is None:
        return np.mean(vecs, axis=0)

    poids = np.array(poids).reshape(-1, 1)
    return np.sum(vecs * poids, axis=0) / np.sum(poids)

In [4]:
def get_vecteurs_Profil_User(user_id: str, conn=conn):
    df_vecteurs_profil_user = pd.read_sql_query("""
                SELECT E.categorie, E.text_initial, E.vecteur FROM Embeddings AS E 
                JOIN Profil_Embeddings AS PE ON E.embeddings_id = PE.embeddings_id
                JOIN Users AS U ON PE.user_id = U.user_id 
                WHERE U.user_id = '""" + user_id + "';"
                , conn)
    # Convertir les vecteurs stockés en BLOB en tableaux numpy
    df_vecteurs_profil_user['vecteur'] = df_vecteurs_profil_user['vecteur'].apply(lambda x: np.frombuffer(x, dtype=np.float32))
    
    # Tableaux avec tous les textes
    liste_hobby = df_vecteurs_profil_user[df_vecteurs_profil_user['categorie'] == 'hobby']['text_initial'].to_list()
    liste_trait = df_vecteurs_profil_user[df_vecteurs_profil_user['categorie'] == 'trait']['text_initial'].to_list()
    liste_job = df_vecteurs_profil_user[df_vecteurs_profil_user['categorie'] == 'metier']['text_initial'].to_list()
    
    # Calculer les vecteurs moyens pondérés pour chaque catégorie
    moy_vect_hobby = moyenne_ponderee_embeddings(df_vecteurs_profil_user[df_vecteurs_profil_user['categorie'] == 'hobby']['vecteur'].to_list())
    moy_vect_trait = moyenne_ponderee_embeddings(df_vecteurs_profil_user[df_vecteurs_profil_user['categorie'] == 'trait']['vecteur'].to_list())
    moy_vect_job = moyenne_ponderee_embeddings(df_vecteurs_profil_user[df_vecteurs_profil_user['categorie'] == 'metier']['vecteur'].to_list()) # Je le fait pour le type de sortie str -> np.array
    
    # Retourner les vecteurs moyens et le vecteur métier
    return liste_hobby, moy_vect_hobby, liste_trait, moy_vect_trait, liste_job, moy_vect_job

In [5]:
def normalisation_des_vecteurs(vecteurs: pd.Series):
    X = np.vstack(vecteurs).astype(np.float32)
    faiss.normalize_L2(X)
    return list(X)

In [6]:
# Récupération des données des utilisateurs
df_Users = pd.read_sql_query("SELECT * FROM Users", conn)
print(df_Users.shape)
df_Users.head()

(10002, 7)


,user_id,email,password_hash,nom,prenom,age,genre
0,54da33db-d1ec-4eb7-8221-db9749969d97,christine.lelièvre@mail.com,2917c4d6a4b1609f93fce80fc32b4ad264d3e1a4c6cf99...,Lelièvre,Christine,33,M
1,43a22bcd-d5b7-49f1-9077-021de14e2f9a,susan.techer@mail.com,894e19ee48abc93a803b06b2a9ab7bd867480e8e4a8172...,Techer,Susan,47,F
2,98ca72c8-1bdb-4c70-a9cf-8b8db52356eb,frédéric.gros@mail.com,b9cd36c0fca95ec7ae7a22b9dffe0a8ef61a62ea36a7ae...,Gros,Frédéric,65,M
3,627db46a-10bf-4d16-93a4-b802510c37c4,olivier.menard@mail.com,89f1e45c16530948f860447ea4bd5d68b6c3c2559cd6c4...,Menard,Olivier,65,M
4,24c98757-91e3-4510-8242-c983eff0ece3,suzanne.grondin@mail.com,a0aa4a016a344554b7d42ca12a3bb89c7785e99405d6d0...,Grondin,Suzanne,41,F


## Mise en forme des données
---

In [7]:
# On garde uniquement les 10 000 premiers utilisateurs
df_Users = df_Users[:10_000] 

In [8]:
# Calculer et ajouter les vecteurs de profil pour chaque utilisateur
df_Users["liste_hobby"], df_Users["vecteur_hobby"], df_Users["liste_trait"], df_Users["vecteur_trait"], df_Users["liste_metier"], df_Users["vecteur_metier"] = zip(*df_Users["user_id"].apply(get_vecteurs_Profil_User))
# Temps (10_000) : 3m

In [9]:
# Normalisation des vecteurs
df_Users["vecteur_hobby"] = normalisation_des_vecteurs(df_Users["vecteur_hobby"].values)
df_Users["vecteur_trait"] = normalisation_des_vecteurs(df_Users["vecteur_trait"].values)
df_Users["vecteur_metier"] = normalisation_des_vecteurs(df_Users["vecteur_metier"].values)
# Temps (10_000) : 1s

In [10]:
# Pas forcement utile car on filtre avant le matching
# Tokenisation de la colonne genre
# df_Users['genre_token'] = df_Users['genre'].apply(lambda x: 1 if x == 'F' else 0)
# Temps (10_000) : 0.5s

In [11]:
# Ajout d'un identifiant FAISS pour chaque utilisateur car FAISS nécessite des identifiants entiers
df_Users["faiss_id"] = df_Users.index.astype(np.int64)

In [12]:
print(df_Users.dtypes)
print()
print(df_Users.shape)
df_Users.head()

user_id           object
email             object
password_hash     object
nom               object
prenom            object
age                int64
genre             object
liste_hobby       object
vecteur_hobby     object
liste_trait       object
vecteur_trait     object
liste_metier      object
vecteur_metier    object
faiss_id           int64
dtype: object

(10000, 14)


,user_id,email,password_hash,nom,prenom,age,genre,liste_hobby,vecteur_hobby,liste_trait,vecteur_trait,liste_metier,vecteur_metier,faiss_id
0,54da33db-d1ec-4eb7-8221-db9749969d97,christine.lelièvre@mail.com,2917c4d6a4b1609f93fce80fc32b4ad264d3e1a4c6cf99...,Lelièvre,Christine,33,M,"[Cyclisme sur route, Voile, Streaming, Cyclism...","[0.07127534, 0.0025000903, -0.03623265, -0.017...","[Loyal, Négatif, Perfectionniste, Conservateur]","[0.026794832, 0.060765605, -0.07501805, -0.031...",[Chaudronnier],"[-0.043414973, 0.024829665, -0.03514247, 0.065...",0
1,43a22bcd-d5b7-49f1-9077-021de14e2f9a,susan.techer@mail.com,894e19ee48abc93a803b06b2a9ab7bd867480e8e4a8172...,Techer,Susan,47,F,"[Flûte, Bachata, Aïkido, Yoga]","[0.037167076, -0.03485525, -0.047245704, 0.062...","[Agressif, Discret]","[0.045850296, 0.06483497, -0.039785773, 0.0227...",[Orthodontiste],"[0.023546293, 0.06853952, -0.044546444, 0.0881...",1
2,98ca72c8-1bdb-4c70-a9cf-8b8db52356eb,frédéric.gros@mail.com,b9cd36c0fca95ec7ae7a22b9dffe0a8ef61a62ea36a7ae...,Gros,Frédéric,65,M,"[Cybersécurité, Harmonica]","[-0.022134779, -0.01912512, -0.06573106, -0.09...","[Coopératif, Empathique]","[0.012761186, -0.0331161, -0.045033492, 0.0419...",[Comptable],"[0.056332674, 0.0058492585, -0.06949069, -0.04...",2
3,627db46a-10bf-4d16-93a4-b802510c37c4,olivier.menard@mail.com,89f1e45c16530948f860447ea4bd5d68b6c3c2559cd6c4...,Menard,Olivier,65,M,"[Batterie, Flûte, Basket-ball]","[-0.005895206, 0.092840545, -0.03179556, -0.02...",[Rigide],"[-0.030995738, -0.0007928369, -0.0393875, -0.0...",[Acteur],"[-0.017877122, -0.06964087, -0.032792557, -0.0...",3
4,24c98757-91e3-4510-8242-c983eff0ece3,suzanne.grondin@mail.com,a0aa4a016a344554b7d42ca12a3bb89c7785e99405d6d0...,Grondin,Suzanne,41,F,"[Chorale, Scénarisation, Dessin, Électronique,...","[0.004016554, 0.05062951, -0.012900323, 0.0024...","[Autoritaire, Perfectionniste, Patient, Agress...","[0.0081748795, 0.053316258, -0.041150734, 0.00...",[Orthophoniste],"[0.03596259, 0.025672827, 0.015121806, 0.02040...",4


## Création du modèle
---

In [13]:
user_id = "a52f0e58-a88c-4098-846b-5af80398ac87" # Yann
# user_id = "de7e320d-00f3-4161-a7dc-eb78ea6e92a9" # Elodie

In [14]:
# Récupération des données des utilisateurs
df_Envie_Users = pd.read_sql_query("SELECT E.user_id, E.envies_id, E.Age_min, E.Age_max, E.genre FROM Users AS U JOIN Envies AS E ON U.user_id = E.user_id WHERE U.user_id = '" + user_id + "'", conn)
print(df_Envie_Users.shape)
df_Envie_Users.head()

(1, 5)


,user_id,envies_id,Age_min,Age_max,genre
0,a52f0e58-a88c-4098-846b-5af80398ac87,1,20,25,F


In [15]:
def get_vecteurs_Envie_User(user_id: str, conn=conn):
    df_vecteurs_envie_user = pd.read_sql_query("""
                SELECT E.categorie, E.text_initial, E.vecteur FROM Embeddings AS E 
                JOIN Envies_Embeddings AS EE ON E.embeddings_id = EE.embeddings_id
                JOIN Envies AS U ON EE.envies_id = U.envies_id 
                WHERE U.user_id = '""" + user_id + "';"
                , conn)
    # Convertir les vecteurs stockés en BLOB en tableaux numpy
    df_vecteurs_envie_user['vecteur'] = df_vecteurs_envie_user['vecteur'].apply(lambda x: np.frombuffer(x, dtype=np.float32))
    
    # Tableaux avec tous les textes
    liste_hobby = df_vecteurs_envie_user[df_vecteurs_envie_user['categorie'] == 'hobby']['text_initial'].to_list()
    liste_trait = df_vecteurs_envie_user[df_vecteurs_envie_user['categorie'] == 'trait']['text_initial'].to_list()
    liste_job = df_vecteurs_envie_user[df_vecteurs_envie_user['categorie'] == 'metier']['text_initial'].to_list()
    
    # Calculer les vecteurs moyens pondérés pour chaque catégorie
    moy_vect_hobby = moyenne_ponderee_embeddings(df_vecteurs_envie_user[df_vecteurs_envie_user['categorie'] == 'hobby']['vecteur'].to_list())
    moy_vect_trait = moyenne_ponderee_embeddings(df_vecteurs_envie_user[df_vecteurs_envie_user['categorie'] == 'trait']['vecteur'].to_list())
    moy_vect_job = moyenne_ponderee_embeddings(df_vecteurs_envie_user[df_vecteurs_envie_user['categorie'] == 'metier']['vecteur'].to_list()) # Je le fait pour le type de sortie str -> np.array
    
    # Retourner les vecteurs moyens et le vecteur métier
    return liste_hobby, moy_vect_hobby, liste_trait, moy_vect_trait, liste_job, moy_vect_job

#### Nettoyage du vecteur d'envie de l'utilisateur
---

Pour qu'il correspond aux vecteurs utilisateur

In [16]:
# Calculer et ajouter les vecteurs d'envie de l'utilisateur
df_Envie_Users["liste_hobby"], df_Envie_Users["vecteur_hobby"], df_Envie_Users["liste_trait"], df_Envie_Users["vecteur_trait"], df_Envie_Users["liste_metier"], df_Envie_Users["vecteur_metier"] = zip(*df_Envie_Users["user_id"].apply(get_vecteurs_Envie_User))

# Normalisation des vecteurs
df_Envie_Users["vecteur_hobby"] = normalisation_des_vecteurs(df_Envie_Users["vecteur_hobby"].values)
df_Envie_Users["vecteur_trait"] = normalisation_des_vecteurs(df_Envie_Users["vecteur_trait"].values)
df_Envie_Users["vecteur_metier"] = normalisation_des_vecteurs(df_Envie_Users["vecteur_metier"].values)

# Pas forcement utile car on filtre avant le matching
# Tokenisation de la colonne genre
# df_Envie_Users['genre_token'] = df_Envie_Users['genre'].apply(lambda x: 1 if x == 'F' else 0)

In [17]:
print(df_Envie_Users.dtypes)
print()
print(df_Envie_Users.shape)
df_Envie_Users.head()

user_id           object
envies_id          int64
Age_min            int64
Age_max            int64
genre             object
liste_hobby       object
vecteur_hobby     object
liste_trait       object
vecteur_trait     object
liste_metier      object
vecteur_metier    object
dtype: object

(1, 11)


,user_id,envies_id,Age_min,Age_max,genre,liste_hobby,vecteur_hobby,liste_trait,vecteur_trait,liste_metier,vecteur_metier
0,a52f0e58-a88c-4098-846b-5af80398ac87,1,20,25,F,"[Intelligence artificielle, Sculpture, Golf, C...","[0.031128218, 0.043568797, -0.034548193, -0.02...","[Impulsif, Honnête, Enthousiaste, Motivé, Into...","[0.0492927, 0.033372696, -0.05250917, 0.006664...","[Livreur, Magasinier]","[-0.021540193, 0.040092025, -0.054153927, 0.07..."


## Fonction du modèle
---

On fait le filtre sur le genre et sur la tranche d'âge rechercher avant de faire le modèle

In [18]:
def search(index: faiss.IndexIDMap, envie_vec, k: int = 5) -> pd.DataFrame:
    q = envie_vec.astype(np.float32).reshape(1, -1)

    # k nearest neighbors (run du modèle Faiss)
    D, I = index.search(q, k)

    # Récupérer les informations des utilisateurs correspondants
    results = df_Users.iloc[I[0]][["faiss_id", "nom", "prenom", "age", "genre", "liste_hobby", "liste_trait", "liste_metier"]].copy()
    results["score"] = D[0] # Ajouter les scores de similarité

    return results

In [19]:
# Filtrage des utilisateurs selon les envies de l'utilisateur
df_filtrer = df_Users[df_Users["genre"] == df_Envie_Users["genre"].values[0]]
df_filtrer = df_filtrer[df_filtrer["age"] <= df_Envie_Users["Age_max"].values[0]]
df_filtrer = df_filtrer[df_filtrer["age"] >= df_Envie_Users["Age_min"].values[0]]
print(df_filtrer.shape)
df_filtrer.head()

(456, 14)


,user_id,email,password_hash,nom,prenom,age,genre,liste_hobby,vecteur_hobby,liste_trait,vecteur_trait,liste_metier,vecteur_metier,faiss_id
14,c5b9d5f5-d29d-44a9-bd58-3248993cbeb6,constance.verdier@mail.com,d3c93224299b972edee8f34f74dd568ac5d1da87dfbd5c...,Verdier,Constance,24,F,"[Escrime, Kizomba, Intelligence artificielle, ...","[-0.016090984, 0.06774993, -0.074718066, 0.006...","[Spontané, Aimable, Déterminé, Aimable, Organisé]","[0.029868908, 0.0018446132, -0.03231706, -0.00...",[Médecin spécialiste],"[-0.05554558, 0.0154114645, -0.026841002, 0.03...",14
25,0ecebab7-4059-471d-b4f9-8e3d6a3eeb57,jérôme.bourgeois@mail.com,47b8a4733e971a75f869586f69178ee2edb47da2cef620...,Bourgeois,Jérôme,25,F,"[Tricot, Décoration intérieure, Programmation]","[0.001569999, 0.010278673, -0.02136029, -0.024...","[Compétitif, Réservé, Méthodique, Autoritaire,...","[0.007156047, 0.07149648, -0.04291228, 0.00844...",[Chauffeur routier],"[-0.0056866254, 0.05025225, -0.04505707, 0.051...",25
63,4e83b59d-d79a-4461-ad5e-67fba931cc6b,julie.courtois@mail.com,7144cc3c72e87449657e7933a5d5f196651f7c4e74b51e...,Courtois,Julie,20,F,"[Jeux de rôle, Romans, Taekwondo, Cocktails, C...","[-0.007284499, 0.045569245, -0.0134427, 0.0127...","[Généreux, Instable, Bienveillant]","[0.030427003, 0.01080061, -0.02085732, 0.01522...",[Aide-soignant],"[-0.0075356853, -0.06604484, 0.028867722, 0.06...",63
76,79b98855-2112-4141-9b75-e1732ebc0c58,théodore.royer@mail.com,e819f18f18212820bc832ac5e73cd5101fab9015daac2b...,Royer,Théodore,21,F,"[Danse contemporaine, Bachata, Boxe, Sound des...","[0.04642031, -0.012761939, -0.03038248, -0.003...",[Consciencieux],"[0.087919414, 0.02495864, -0.06286995, -0.0031...",[Chef de rang],"[0.02568455, 0.053258587, -0.0107880235, -0.00...",76
99,3540befe-1962-44bf-9638-322f7b26b81b,jérôme.besson@mail.com,05ed516c195dd2b6c8a3fc47f0d6b15988bba07edb7475...,Besson,Jérôme,21,F,[Philosophie],"[0.014526393, 0.10013223, -0.076151416, 0.0522...",[Analytique],"[-0.04614174, 0.07579126, -0.070000485, -0.013...",[Artiste plasticien],"[-0.035559606, -0.042097837, 0.020379016, 0.06...",99


#### Modèle des hobbies
---

In [20]:
# Creation de l'index Faiss avec les vecteurs filtrés
index_hobby = faiss.IndexIDMap(faiss.IndexFlatIP(384))

X_hobby = np.vstack(df_filtrer["vecteur_hobby"].values).astype(np.float32)
index_hobby.add_with_ids(X_hobby, df_filtrer["faiss_id"].values)

In [21]:
res_hobby = search(index_hobby, df_Envie_Users["vecteur_hobby"].values[0], k=10)
print(res_hobby.shape)
res_hobby.head()

(10, 9)


,faiss_id,nom,prenom,age,genre,liste_hobby,liste_trait,liste_metier,score
535,535,Richard,Édith,20,F,"[Danse contemporaine, Clarinette, Observation ...",[Anxieux],[UX designer],0.831987
7763,7763,Guyon,Louis,22,F,"[Cor, Poterie, Basse, Tissage, Escalade]","[Rigide, Authentique, Solitaire]",[Scrum master],0.830456
5538,5538,Girard,Manon,23,F,"[Ukulélé, Survie, Psychologie, Écriture, Volle...","[Solitaire, Agressif, Manipulateur]",[Professeur des écoles],0.817151
6149,6149,Ruiz,Agathe,24,F,"[Broderie, Basket-ball, Canoë-kayak, Broderie,...",[Réfléchi],[Psychologue],0.810506
345,345,De Sousa,Capucine,24,F,"[Calligraphie, Poterie, Natation]",[Minutieux],[Livreur],0.807607


#### Modèle des traits de caractères
---

In [22]:
# Creation de l'index Faiss avec les vecteurs filtrés
index_traits = faiss.IndexIDMap(faiss.IndexFlatIP(384))

X_traits = np.vstack(df_filtrer["vecteur_trait"].values).astype(np.float32)
index_traits.add_with_ids(X_traits, df_filtrer["faiss_id"].values)

In [23]:
res_traits = search(index_traits, df_Envie_Users["vecteur_trait"].values[0], k=10)
print(res_traits.shape)
res_traits.head()

(10, 9)


,faiss_id,nom,prenom,age,genre,liste_hobby,liste_trait,liste_metier,score
5049,5049,Grondin,Eugène,20,F,"[Illustration numérique, Canoë-kayak, Botanique]","[Déterminé, Discret, Indécis, Inconstant, Altr...",[Compositeur],0.928315
5286,5286,Lecomte,Margaud,23,F,"[Mangas, Jeux de société, Figurines, Illustrat...","[Instable, Manipulateur, Bienveillant, Honnête...",[Gestionnaire de patrimoine],0.927037
6647,6647,Rodriguez,Mathilde,20,F,"[Taekwondo, Saxophone, Sudoku, Guitare, Stop m...","[Réfléchi, Optimiste, Impulsif]",[Conseiller assurance],0.925482
1773,1773,Guichard,Édith,21,F,[Piano],"[Indépendant, Consciencieux, Persévérant, Sinc...",[Enseignant-chercheur],0.924323
5100,5100,Pascal,Philippine,23,F,"[DJing, Botanique, Rugby, Intelligence artific...","[Déterminé, Réfléchi, Indécis, Motivé]",[QA engineer],0.924106


#### Modèle des métiers
---

In [24]:
# Creation de l'index Faiss avec les vecteurs filtrés
index_metier = faiss.IndexIDMap(faiss.IndexFlatIP(384))

X_metier = np.vstack(df_filtrer["vecteur_metier"].values).astype(np.float32)
index_metier.add_with_ids(X_metier, df_filtrer["faiss_id"].values)

In [25]:
res_metier = search(index_metier, df_Envie_Users["vecteur_metier"].values[0], k=10)
print(res_metier.shape)
res_metier.head()

(10, 9)


,faiss_id,nom,prenom,age,genre,liste_hobby,liste_trait,liste_metier,score
9360,9360,Bonnin,Diane,25,F,[Aïkido],"[Optimiste, Rigide]",[Magasinier],0.914897
4174,4174,Boulay,Margaux,24,F,[Métallerie],"[Rigide, Analytique, Introverti]",[Magasinier],0.914897
7818,7818,Schneider,Suzanne,23,F,"[Théâtre, Alto, Volley-ball, Tennis]","[Calme, Discipliné, Colérique, Endurant]",[Livreur],0.871867
372,372,Lefèvre,Geneviève,25,F,"[Poterie, Escrime, Astronomie, Aviron]","[Vaniteux, Indiscipliné, Patient, Courageux, C...",[Livreur],0.871867
345,345,De Sousa,Capucine,24,F,"[Calligraphie, Poterie, Natation]",[Minutieux],[Livreur],0.871867


#### Concaténation des resultats
---

In [26]:
df_resultats_final = pd.concat([res_hobby, res_traits, res_metier], axis=0, ignore_index=True)
df_resultats_final.drop_duplicates(subset=["faiss_id"], keep="first", inplace=True)
df_resultats_final = df_resultats_final.sort_values(by="score", ascending=False).reset_index(drop=True)
print(df_resultats_final.shape)
df_resultats_final.head()

(29, 9)


,faiss_id,nom,prenom,age,genre,liste_hobby,liste_trait,liste_metier,score
0,5049,Grondin,Eugène,20,F,"[Illustration numérique, Canoë-kayak, Botanique]","[Déterminé, Discret, Indécis, Inconstant, Altr...",[Compositeur],0.928315
1,5286,Lecomte,Margaud,23,F,"[Mangas, Jeux de société, Figurines, Illustrat...","[Instable, Manipulateur, Bienveillant, Honnête...",[Gestionnaire de patrimoine],0.927037
2,6647,Rodriguez,Mathilde,20,F,"[Taekwondo, Saxophone, Sudoku, Guitare, Stop m...","[Réfléchi, Optimiste, Impulsif]",[Conseiller assurance],0.925482
3,1773,Guichard,Édith,21,F,[Piano],"[Indépendant, Consciencieux, Persévérant, Sinc...",[Enseignant-chercheur],0.924323
4,5100,Pascal,Philippine,23,F,"[DJing, Botanique, Rugby, Intelligence artific...","[Déterminé, Réfléchi, Indécis, Motivé]",[QA engineer],0.924106
